# 04. Longitudinal Cohort Retention & Churn Decay Analysis

### Project: E-Commerce Customer Funnel & Cohort Retention Analytics

**Objective:** Model monthly customer retention trajectories by segmenting customer bases into acquisition cohorts, tracking repeat transaction frequency over successive activity cycles, computing retention percentage decay, and constructing heatmap visualizations for lifecycle reporting.

## 1. Environment Configuration & Sanitized Data Ingestion

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('data/retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Ingested {len(df):,} sanitized transactions.')

## 2. Acquisition Cohort Identification

Each customer is assigned to an acquisition cohort based on the calendar month of their first recorded transaction (`CohortMonth`).

In [ ]:
# Extract monthly period per transaction
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Determine customer acquisition cohort
customer_cohorts = df.groupby('CustomerID')['InvoiceMonth'].min().reset_index()
customer_cohorts.rename(columns={'InvoiceMonth': 'CohortMonth'}, inplace=True)

# Append acquisition cohort to transactional records
df = df.merge(customer_cohorts, on='CustomerID')
df[['CustomerID', 'Invoice', 'InvoiceDate', 'InvoiceMonth', 'CohortMonth']].head()

## 3. Elapsed Cycle Derivation (`CohortIndex`)

The `CohortIndex` quantifies the number of calendar months elapsed between a customer's acquisition month (`CohortMonth`) and subsequent transaction months (`InvoiceMonth`):
- `CohortIndex = 0`: In-cohort acquisition transaction.
- `CohortIndex = 1`: Return transaction 1 month post-acquisition.

In [ ]:
# Calculate elapsed months between acquisition and order
df['CohortIndex'] = (
    (df['InvoiceMonth'].dt.year - df['CohortMonth'].dt.year) * 12 +
    (df['InvoiceMonth'].dt.month - df['CohortMonth'].dt.month)
)

print(f'Observed tracking horizon: 0 to {df["CohortIndex"].max()} months.')
df[['CustomerID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head()

## 4. Active Customer Activity Matrix Construction

In [ ]:
# Aggregate distinct active customers by cohort and elapsed cycle
cohort_grouped = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_grouped.rename(columns={'CustomerID': 'ActiveCustomers'}, inplace=True)

# Pivot into triangular matrix
cohort_pivot = cohort_grouped.pivot_table(
    index='CohortMonth',
    columns='CohortIndex',
    values='ActiveCustomers'
)

print('Raw Active Customer Volume (First 8 Cohorts, First 6 Cycles):')
cohort_pivot.iloc[:8, :7]

## 5. Retention Percentage Normalization

Active customer volumes are indexed against initial cohort acquisition volume (`CohortIndex = 0`), establishing retention rate decay curves across successive monthly intervals.

In [ ]:
# Normalize by base cohort size (Column 0)
cohort_sizes = cohort_pivot[0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0).round(4) * 100

print('Retention Matrix in Percentage (First 8 Cohorts, First 6 Cycles):')
retention_matrix.iloc[:8, :7].round(1)

## 6. Retention Heatmap Visualization

In [ ]:
plt.figure(figsize=(14, 8))

sns.heatmap(
    retention_matrix.iloc[:16, :13],
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    vmin=0,
    vmax=50,
    linewidths=0.5,
    linecolor='#ffffff',
    cbar_kws={'label': 'Retention Rate (%)'}
)

plt.title('Monthly Cohort Retention Matrix (Decay Curves)', fontsize=13, pad=15)
plt.xlabel('Elapsed Activity Cycles (Months Post-Acquisition)', fontsize=10)
plt.ylabel('Acquisition Cohort', fontsize=10)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Performance Benchmarks & Artifact Serialization

In [ ]:
avg_m1 = retention_matrix[1].mean()
avg_m3 = retention_matrix[3].mean()
avg_m6 = retention_matrix[6].mean()

print('Portfolio Retention Benchmarks:')
print(f'- Month 1 Mean Retention: {avg_m1:.1f}%')
print(f'- Month 3 Mean Retention: {avg_m3:.1f}%')
print(f'- Month 6 Mean Retention: {avg_m6:.1f}%')

# Export analytical datasets
retention_matrix.to_csv('data/cohort_retention_matrix.csv')
cohort_grouped.to_csv('data/cohort_counts.csv', index=False)
print('\nSaved data/cohort_retention_matrix.csv and data/cohort_counts.csv.')